# Week 5: Final model, test evaluation, analysis and demo

1. The final model is chosen from validation results only.
2. The test set is evaluated once and then does not run again.  
3. Every other analysis here uses the **validation** set.
4. Everything is saved to `fruit_project/final/`.

| Section | What it does |
|---|---|
| 1-3 | Setup, shared code, evaluation and Grad-CAM functions (identical to `03_pipeline.ipynb`) |
| 4 | Final model decision and copy its checkpoint to `final/` |
| 5 | Test evaluation |
| 6 | Grad-CAM review sample |
| 7 | Camera-shortcut check |
| 8 | Unsure threshold |
| 9 | Robustness to dark, bright, blurred and low-resolution photos |
| 10 | Model size and speed |
| 11 | Non-apple photos |
| 12 | Gradio demo |

## 1. Setup (run at the start of every session)

In [ ]:
# 1. Setup
!pip install grad-cam --quiet

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json, random, time, csv
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

PROJECT = "/content/drive/MyDrive/fruit_project"
DATA = "/content/dataset"
if not os.path.isdir(DATA):
    shutil.unpack_archive(f"{PROJECT}/dataset.zip", DATA)     # takes a minute or two

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
assert device.type == "cuda", "Runtime -> Change runtime type -> GPU, then rerun everything"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

split = json.load(open(f"{PROJECT}/split.json"))
probe = datasets.ImageFolder(DATA)
assert probe.class_to_idx == split["class_to_idx"], "class order changed"
assert len(probe) == sum(len(split[k]) for k in ["train", "val", "test"]), "image count mismatch"

class_names = probe.classes
num_classes = len(class_names)
HEALTHY = probe.class_to_idx["Healthy"]
for sub in ["checkpoints", "results"]:
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)
print("OK:", probe.class_to_idx, len(probe), "images")

## 2. Shared code

In [ ]:
# 2. Shared code
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet statistics
EVAL_TFM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def make_datasets(train_tfm):
    train_full = datasets.ImageFolder(DATA, transform=train_tfm)   # training images may be augmented
    eval_full = datasets.ImageFolder(DATA, transform=EVAL_TFM)     # validation images never augmented
    return Subset(train_full, split["train"]), Subset(eval_full, split["val"])

def make_loaders(train_ds, val_ds, bs=32):
    return (DataLoader(train_ds, bs, shuffle=True, num_workers=2),
            DataLoader(val_ds, bs, shuffle=False, num_workers=2))

# Class weights come from the train split only. Rarer classes count for more in the loss.
train_targets = np.array(probe.targets)[split["train"]]
counts = np.bincount(train_targets, minlength=num_classes)
class_weights = torch.tensor(len(train_targets) / (num_classes * counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("training images per class:", dict(zip(class_names, counts.tolist())))

def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)   # pretrained on ImageNet
    for p in model.parameters():
        p.requires_grad = False                                        # freeze everything
    model.fc = nn.Linear(model.fc.in_features, num_classes)            # new trainable head
    return model.to(device)

def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model = build_model(len(ckpt["class_names"]))
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval()

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over the data. Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

def predict_all(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p.numpy()); ps.append(p.argmax(1).numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(probs)

## 3. Evaluation and Grad-CAM functions

In [ ]:
# 5. Evaluation and logging
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

def plot_history(h, save_path=None):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["train_acc"], label="train"); ax[1].plot(h["val_acc"], label="val")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

def evaluate_run(model, loader, class_names, save_prefix):
    y, p, _ = predict_all(model, loader)
    print(classification_report(y, p, target_names=class_names, digits=3))

    cm = confusion_matrix(y, p, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax[0], cmap="Blues", xticks_rotation=45, colorbar=False)
    ax[0].set_title("Image counts")
    ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=class_names).plot(
        ax=ax[1], cmap="Blues", xticks_rotation=45, colorbar=False, values_format=".2f")
    ax[1].set_title("Share of each true class")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_confusion.png", dpi=150)
    plt.show()

    recall = np.diag(cm) / cm.sum(axis=1)
    off = cm.copy(); np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    diseased, healthy = (y != HEALTHY), (y == HEALTHY)
    metrics = {
        "val_acc": float((y == p).mean()),
        "val_macro_f1": float(f1_score(y, p, average="macro")),
        "miss_rate": float(((p == HEALTHY) & diseased).sum() / diseased.sum()),
        "false_alarm_rate": float(((p != HEALTHY) & healthy).sum() / healthy.sum()),
        "weakest_class": class_names[int(recall.argmin())],
        "weakest_recall": float(recall.min()),
        "most_confused": f"{class_names[i]} -> {class_names[j]} ({off[i, j]} images)",
    }
    print(f"accuracy {metrics['val_acc']:.3f} | macro-F1 {metrics['val_macro_f1']:.3f}")
    print(f"miss rate (diseased called Healthy): {metrics['miss_rate']:.3f} | "
          f"false alarms (Healthy called diseased): {metrics['false_alarm_rate']:.3f}")
    print(f"weakest class: {metrics['weakest_class']} (recall {metrics['weakest_recall']:.3f}) | "
          f"most confused: {metrics['most_confused']}")
    return metrics

LOG_COLUMNS = ["run", "augmentation", "finetune_mode", "epochs", "best_epoch", "seed", "sec_per_epoch",
               "val_acc", "val_macro_f1", "miss_rate", "false_alarm_rate",
               "weakest_class", "weakest_recall", "most_confused", "notes"]

def log_result(run, aug, mode, epochs, best_epoch, seed, sec_per_epoch, metrics, notes=""):
    path = f"{PROJECT}/results/experiments.csv"
    new = not os.path.exists(path)
    row = {"run": run, "augmentation": aug, "finetune_mode": mode, "epochs": epochs,
           "best_epoch": best_epoch, "seed": seed, "sec_per_epoch": round(sec_per_epoch, 1), "notes": notes}
    row.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()})
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if new:
            w.writeheader()
        w.writerow(row)

In [ ]:
# 6. Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def misclassified_indices(model, dataset, bs=64):
    """Positions in `dataset` that the model gets wrong. Handy for studying failures."""
    loader = DataLoader(dataset, batch_size=bs, shuffle=False, num_workers=2)
    y, p, _ = predict_all(model, loader)
    return np.where(y != p)[0].tolist()

def gradcam_grid(model, dataset, class_names, indices=None, n=6, save_path=None, seed=0):
    # Grad-CAM needs gradients to flow through the whole network. With a frozen backbone they would not,
    # so we switch them back on.
    for p in model.parameters():
        p.requires_grad = True
    model.eval()
    cam = GradCAM(model=model, target_layers=[model.layer4[-1]])      # last conv block of ResNet18

    if indices is None:
        indices = random.Random(seed).sample(range(len(dataset)), n)
    indices = indices[:n]
    if len(indices) == 0:
        print("nothing to show")
        return

    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6.6), squeeze=False)
    for col, i in enumerate(indices):
        x, y = dataset[i]
        inp = x.unsqueeze(0).to(device)
        probs = torch.softmax(model(inp), dim=1)[0].detach()
        pred = int(probs.argmax())
        targets = [ClassifierOutputTarget(pred)]                       # explain the predicted class, explicitly
        heat = cam(input_tensor=inp, targets=targets)[0]
        rgb = (x.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1).astype(np.float32)   # undo normalisation

        axes[0][col].imshow(rgb); axes[0][col].axis("off")
        axes[0][col].set_title(f"true: {class_names[y]}", fontsize=9)
        axes[1][col].imshow(show_cam_on_image(rgb, heat, use_rgb=True)); axes[1][col].axis("off")
        axes[1][col].set_title(f"pred: {class_names[pred]} ({probs[pred]:.0%})", fontsize=9,
                               color="green" if pred == y else "red")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

## 4. The final model

baseline_none_frozen

The frozen ResNet18 baseline (only the last layer is trained, no augmentation, trained for 10 epochs) was chosen.

Reasoning: We compared three ways of training: frozen, last block unfrozen, and all layers unfrozen. Their scores were almost the same (macro-F1 of 0.939, 0.944 and 0.944), and the gaps were smaller than the natural run-to-run change of about 0.009. The share of diseased fruit called healthy was also almost the same (4.5%, 4.5% and 3.7%, which is about two photos apart). Due to a tie, the simplest model was chose.


In [ ]:
# 4. Lock the final model
FINAL_NAME = "baseline_none_frozen"
assert FINAL_NAME != "FILL_IN_RUN_NAME", "Type the chosen run name above first"

src = f"{PROJECT}/checkpoints/{FINAL_NAME}.pt"
assert os.path.exists(src), f"No checkpoint named {FINAL_NAME}"
os.makedirs(f"{PROJECT}/final", exist_ok=True)
FINAL_MODEL = f"{PROJECT}/final/final_model.pt"
INFO = f"{PROJECT}/final/final_model_info.json"

if os.path.exists(FINAL_MODEL):
    old = json.load(open(INFO))
    if old["run"] != FINAL_NAME:
        raise RuntimeError(f"final_model.pt already holds '{old['run']}', not '{FINAL_NAME}'. "
                           f"Resolve this as a team before going on.")
    print("final_model.pt already locked:", old["run"])
else:
    shutil.copy2(src, FINAL_MODEL)
    ckpt = torch.load(src, map_location="cpu")
    json.dump({"run": FINAL_NAME, "config": ckpt.get("config"), "epoch": ckpt["epoch"],
               "val_loss": ckpt["val_loss"], "val_acc": ckpt["val_acc"]}, open(INFO, "w"), indent=2)
    print("Locked:", FINAL_NAME)
print(open(INFO).read())

## Validation data and the final model's validation predictions (used by sections 6 to 9)

In [ ]:
# Validation data and predictions
import re
import pandas as pd
from pathlib import Path

eval_full = datasets.ImageFolder(DATA, transform=EVAL_TFM)          # no augmentation
val_ds = Subset(eval_full, split["val"])
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

model = load_model(FINAL_MODEL)
y_val, p_val, probs_val = predict_all(model, val_loader)
print(f"final model on VALIDATION: accuracy {(y_val == p_val).mean():.3f}")

## 5. Test evaluation (ONCE)

This cell saves the test predictions to `final/test_predictions.npz`. Any later test-based numbers (for example the "unsure" threshold in section 8) are computed from that file, not by running the model on test images again.

In [ ]:
# 5. Test evaluation: run once
TEST_RESULT = f"{PROJECT}/final/test_result.json"
assert not os.path.exists(TEST_RESULT), "The test set was already used (test_result.json exists). Do not run this cell again."

test_ds = Subset(eval_full, split["test"])
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

y_test, p_test, probs_test = predict_all(model, test_loader)
np.savez(f"{PROJECT}/final/test_predictions.npz", y=y_test, p=p_test, probs=probs_test,
         files=np.array([os.path.relpath(probe.samples[i][0], DATA) for i in split["test"]]))

metrics_test = evaluate_run(model, test_loader, class_names, f"{PROJECT}/final/test")
json.dump({"run": FINAL_NAME, "metrics": metrics_test, "n_images": len(test_ds),
           "date": time.strftime("%Y-%m-%d %H:%M")}, open(TEST_RESULT, "w"), indent=2)
print("Saved. Report these numbers as they are.")

## 6. Grad-CAM review (validation images)

For each class, a fixed random sample of correctly classified images, plus up to 20 misclassified ones. Fill in `final/gradcam_review.csv` by hand, one row per image, with one of: `lesion`, `background`, `stem_calyx`, `leaf`, `diffuse`, `unclear`.

In [ ]:
# 6. Grad-CAM review sample
REVIEW_PER_CLASS = 10
rng = random.Random(0)
rel = lambda i: os.path.relpath(probe.samples[split["val"][i]][0], DATA)     # file name of val image i
rows = []

for c, cname in enumerate(class_names):
    ok = [i for i in range(len(y_val)) if y_val[i] == c and p_val[i] == c]
    chosen = rng.sample(ok, min(REVIEW_PER_CLASS, len(ok)))
    for half in range(0, len(chosen), 5):
        gradcam_grid(model, val_ds, class_names, indices=chosen[half:half + 5], n=5,
                     save_path=f"{PROJECT}/final/gradcam_{cname.replace(' ', '_')}_correct_{half // 5 + 1}.png")
    rows += [(cname, "correct", rel(i), "", "") for i in chosen]

wrong = np.where(y_val != p_val)[0].tolist()[:20]
print(f"{int((y_val != p_val).sum())} validation images misclassified; reviewing {len(wrong)}")
for half in range(0, len(wrong), 5):
    gradcam_grid(model, val_ds, class_names, indices=wrong[half:half + 5], n=5,
                 save_path=f"{PROJECT}/final/gradcam_wrong_{half // 5 + 1}.png")
rows += [(class_names[y_val[i]], f"wrong (predicted {class_names[p_val[i]]})", rel(i), "", "") for i in wrong]

review_path = f"{PROJECT}/final/gradcam_review.csv"
if os.path.exists(review_path):
    print("gradcam_review.csv already exists. Not overwriting your notes.")
else:
    pd.DataFrame(rows, columns=["true_class", "kind", "file", "heat_on", "notes"]).to_csv(review_path, index=False)
    print("Template saved:", review_path)

## 7. Camera-shortcut check

The class sizes by file-name style suggest the classes may come from different sessions or devices: Healthy photos are almost all timestamp-named, while Black Rot and Powdery Mildew are almost all `IMG_`/`DSC_`-named. The dataset does not say what produced the names, so this is a hypothesis.

If the model learned the camera's look instead of the disease, accuracy should differ by file-name style **within the same class**. Black Pox has both styles, so it is the cleanest test.

In [ ]:
# 7. Accuracy by file-name style within each class (validation)
ts_pat = re.compile(r"\d{8}_\d{6}")
def scheme(fname):
    if ts_pat.search(fname): return "timestamp"
    if fname.startswith("IMG_"): return "IMG_"
    if fname.startswith("DSC_"): return "DSC_"
    return "other"

val_files = [Path(probe.samples[i][0]).name for i in split["val"]]
df_scheme = pd.DataFrame({"class": [class_names[t] for t in y_val],
                          "style": [scheme(n) for n in val_files],
                          "correct": (y_val == p_val)})
table = df_scheme.groupby(["class", "style"])["correct"].agg(n_images="count", accuracy="mean").round(3)
table["reliable"] = table["n_images"] >= 15
table

## 8. Unsure threshold

If the model's confidence is below a threshold, the tool should say that it isn't sure instead of guessing anyway. We chose the threshold from the validation table.  

In [ ]:
# 8. Unsure threshold: choose on validation
def coverage_table(y, p, probs, thresholds):
    conf = probs.max(axis=1)
    rows = []
    for t in thresholds:
        answered = conf >= t
        n = int(answered.sum())
        if n == 0:
            rows.append((t, 0.0, np.nan, np.nan))
            continue
        acc = float((y[answered] == p[answered]).mean())
        diseased = answered & (y != HEALTHY)
        miss = float(((p == HEALTHY) & diseased).sum() / diseased.sum()) if diseased.sum() else np.nan
        rows.append((t, n / len(y), acc, miss))
    return pd.DataFrame(rows, columns=["threshold", "share_answered", "accuracy_when_answered",
                                       "miss_rate_when_answered"]).round(3)

coverage_table(y_val, p_val, probs_val, [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

0.80 was chosen.

In [ ]:
# 8b. Apply the chosen threshold to the saved test predictions
THRESHOLD = 0.80
assert THRESHOLD is not None, "Set THRESHOLD from the validation table first"
d = np.load(f"{PROJECT}/final/test_predictions.npz")
coverage_table(d["y"], d["p"], d["probs"], [0.0, THRESHOLD])

## 9. Robustness

The final model on validation photos made darker, brighter, blurred, or low-resolution, to mimic dusk, glare, a shaky hand and a poor camera.

In [ ]:
# 9. Robustness on validation photos
from sklearn.metrics import f1_score

CORRUPTIONS = {
    "clean": transforms.Compose([]),
    "darker (x0.5)": transforms.ColorJitter(brightness=(0.5, 0.5)),
    "brighter (x1.6)": transforms.ColorJitter(brightness=(1.6, 1.6)),
    "blurred": transforms.GaussianBlur(kernel_size=9, sigma=3.0),
    "low resolution": transforms.Compose([transforms.Resize((56, 56)), transforms.Resize((224, 224))]),
}
rows = []
for name, corr in CORRUPTIONS.items():
    ds_c = Subset(datasets.ImageFolder(DATA, transform=transforms.Compose([corr, EVAL_TFM])), split["val"])
    loader_c = DataLoader(ds_c, batch_size=32, shuffle=False, num_workers=2)
    y, p, _ = predict_all(model, loader_c)
    diseased = y != HEALTHY
    rows.append((name, (y == p).mean(), f1_score(y, p, average="macro"),
                 ((p == HEALTHY) & diseased).sum() / diseased.sum()))
pd.DataFrame(rows, columns=["condition", "accuracy", "macro_f1", "miss_rate"]).round(3)

## 10. Model size and speed



In [ ]:
# 10. Model size and speed
import copy

print(f"checkpoint size: {os.path.getsize(FINAL_MODEL) / 1e6:.1f} MB")

def ms_per_image(m, dev, runs=30):
    m = m.to(dev).eval()
    xx = torch.randn(1, 3, 224, 224).to(dev)
    with torch.no_grad():
        for _ in range(5):
            m(xx)                                   # warm-up
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(runs):
            m(xx)
        if dev.type == "cuda":
            torch.cuda.synchronize()
    return (time.time() - t0) / runs * 1000

print(f"GPU: {ms_per_image(model, torch.device('cuda')):.1f} ms per image")
print(f"CPU: {ms_per_image(copy.deepcopy(model).cpu(), torch.device('cpu'), runs=10):.1f} ms per image")
model = model.to(device)

Viability on other devices:

## 11. Non-apple photos

The model was not taught to recognize images of things that are not apples, so it answers confidently and wrongly on things that are not apples.

In [ ]:
# 11. Non-apple photos
from google.colab import files

uploaded = files.upload()
model.eval()
for fname in uploaded:
    img = Image.open(fname).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(model(EVAL_TFM(img).unsqueeze(0).to(device)), dim=1)[0].cpu()
    top = int(probs.argmax())
    print(f"{fname}: {class_names[top]} ({probs[top]:.0%})")

## 12. Gradio demo

Upload a photo and get the predicted class, the confidence and the Grad-CAM heatmap. `share=True` gives a temporary public link that works only while the Colab session stays open.

In [ ]:
# 12. Gradio demo
!pip install gradio --quiet
import gradio as gr
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

demo_model = load_model(FINAL_MODEL)
for q in demo_model.parameters():
    q.requires_grad = True                      # Grad-CAM needs gradients to flow
demo_cam = GradCAM(model=demo_model, target_layers=[demo_model.layer4[-1]])
UNSURE_BELOW = 0.80

def diagnose(img):
    if img is None:
        return None, "Upload a photo of an apple.", None
    img = img.convert("RGB")
    inp = EVAL_TFM(img).unsqueeze(0).to(device)
    probs = torch.softmax(demo_model(inp), dim=1)[0].detach().cpu()
    top = int(probs.argmax())
    conf = float(probs[top])
    heat = demo_cam(input_tensor=inp, targets=[ClassifierOutputTarget(top)])[0]
    rgb = np.asarray(img.resize((224, 224)), dtype=np.float32) / 255.0
    overlay = show_cam_on_image(rgb, heat, use_rgb=True)
    if conf < UNSURE_BELOW:
        verdict = f"Not sure (best guess: {class_names[top]}, {conf:.0%}). Please ask an expert."
    else:
        verdict = f"{class_names[top]} ({conf:.0%} confident)"
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}, verdict, overlay

demo = gr.Interface(
    fn=diagnose,
    inputs=gr.Image(type="pil", label="Apple photo"),
    outputs=[gr.Label(num_top_classes=len(class_names), label="Prediction"),
             gr.Textbox(label="Verdict"),
             gr.Image(label="Grad-CAM: where the model looked")],
    title="Apple defect checker (prototype)",
    description="Healthy, Black Rot, Black Pox or Powdery Mildew. Trained on one apple variety from one region. "
                "A research prototype, not for real decisions.",
)
demo.launch(share=True)